# Fontes de imagens pré-treinadas

* Roboflow Universe
  * https://universe.roboflow.com/browse/manufacturing
* Kaggle
* Google Open Images v7 dataset

## Datasets selecionados

No Roboflow Universe foi possível encontrar modelos de fotos de caixas empilhadas. Foi adicinado em um novo projeto e dataset v1:

* https://app.roboflow.com/embrapac/products-counter/1

In [1]:
# checar se GPU disponível
!nvidia-smi

Mon Mar 16 09:02:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce MX550           Off |   00000000:01:00.0 Off |                  N/A |
| N/A   49C    P5              4W /   30W |     387MiB /   2048MiB |     14%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Instalando dependências

In [2]:
#!pip uninstall -y numpy ultralytics roboflow
#!pip install roboflow==1.1.48 ultralytics==8.2.103 numpy==1.26.4
!pip install roboflow==1.1.48 ultralytics

In [3]:
from IPython import display
display.clear_output()

# prevent ultralytics from tracking your activity
!yolo settings sync=False

import ultralytics
ultralytics.checks()

Ultralytics 8.4.14 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce MX550, 1682MiB)
Setup complete ✅ (12 CPUs, 15.3 GB RAM, 121.8/460.4 GB disk)


# Obtenção do dataset criado e rotulado no RoboFlow

In [5]:
from roboflow import Roboflow
# from google.colab import userdata

dataset_version = 1
dataset_name=f"exp_small_dataset_v{dataset_version}"
ROBOFLOW_WORKSPACE = "embrapac"
ROBOFLOW_PROJECT = "products-counter"
# ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
ROBOFLOW_API_KEY = 'eunCPcNclF8el2kssWZ2'

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(dataset_version)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to products-counter-1 in yolov8:: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 19702/19702 [00:02<00:00, 8127.74it/s]


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Fase 4: Usar seu Novo Modelo (O Resultado Final)

    Encontre seu Modelo Treinado:

        Após o treinamento terminar, olhe na barra de arquivos à esquerda do Colab.

        Haverá uma nova pasta chamada runs. Navegue dentro dela: runs -> detect -> train.

        Dentro da pasta train, haverá uma pasta weights.

        Dentro de weights, você encontrará o seu tesouro: o arquivo best.pt. Este é o seu modelo treinado, a melhor versão que ele conseguiu criar durante as 25 épocas.

    Baixe o Modelo:

        Clique com o botão direito em best.pt e escolha "Fazer o download". Salve-o no seu computador, na mesma pasta do seu projeto detector.ipynb.

    Atualize seu Notebook de Detecção:

        Abra seu notebook detector.ipynb (o que fizemos anteriormente).

        Vá para a Célula 2 (Configurações do Projeto).

        Mude a linha do MODEL_PATH para apontar para o seu novo modelo:
        codePython

        # Célula 2: Configurações Globais
        ...
        MODEL_PATH = 'best.pt' # ou o nome que você deu ao baixar
        ...

# Treinamento do modelo

In [4]:
from ultralytics import YOLO

#folder = '{}-{}'.format(ROBOFLOW_PROJECT, dataset_version)
dataset_name = 'products-counter-1'

# caminho para o data.yaml baixado pelo Roboflow
#DATA_YAML = f"/content/{folder}/data.yaml"  # ajuste o caminho se necessário
DATA_YAML = '/home/manasouza/Dropbox/Desenvolvimento/CPqD-FIAP_Sistemas_Embarcados/Parte Prática/Projeto/products-counter-1/data.yaml'

# escolhe o modelo base (nano para começar)
#model = YOLO("yolov8n.pt")
model = YOLO("yolov8s.pt")

# treinamento
model.train(
    data=DATA_YAML,
    epochs=35,
    patience=7,          # Aguarda 50 epochs sem melhora antes de parar (early stopping)
    imgsz=640,
    batch=16,
    # --- ORGANIZAÇÃO DOS EXPERIMENTOS ---
    project="embrapac_runs",
    name=dataset_name,
    plots=True,           # Salva gráficos de loss, mAP, etc.
)


New https://pypi.org/project/ultralytics/8.4.22 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce MX550, 1682MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/manasouza/Dropbox/Desenvolvimento/CPqD-FIAP_Sistemas_Embarcados/Parte Prática/Projeto/products-counter-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=35, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mix

Exception in thread Thread-8 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/home/manasouza/development/pyenvs/embrapac/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/home/manasouza/development/pyenvs/embrapac/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/manasouza/development/pyenvs/embrapac/lib/python3.12/site-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_

optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000667, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
: 0% ──────────── 0/1040  4.0s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
WARNING ⚠️ CUDA out of memory with batch=4. Reducing to batch=2 and retrying (3/3).
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 278.9±208.6 MB/s, size: 124.0 KB)
train: Scanning /home/manasouza/Dropbox/Desenvolvimento/CPqD-FIAP_Sistemas_Embarcados/Parte Prática/Projeto/products-counter-1/train/labels.cache... 8319 images, 27 backgrounds, 2 corrupt: 100% ━━━━━━━━━━━━ 8319/8319 1.6Git/s 0.0s
train: /home/manasouza/Dropbox/Desenvolvimento/CPqD-FIAP_Sistemas_Embarcados/Parte Prática/Projeto/products-counter-1/train/images/104844923-amazon-box-outside_JPG.rf.62cdd8a86949f72d39f053b92e002cf0.jpg: ign

Exception in thread Thread-10 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/home/manasouza/development/pyenvs/embrapac/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/home/manasouza/development/pyenvs/embrapac/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/manasouza/development/pyenvs/embrapac/lib/python3.12/site-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage

optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000667, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
: 0% ──────────── 0/2080  3.1s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/35      1.02G      1.059      2.039      1.428          2        640: 100% ━━━━━━━━━━━━ 4159/4159 3.9it/s 17:52<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 0% ──────────── 0/35  0.0s


OutOfMemoryError: CUDA out of memory. Tried to allocate 252.00 MiB. GPU 0 has a total capacity of 1.64 GiB of which 147.12 MiB is free. Including non-PyTorch memory, this process has 1.02 GiB memory in use. Of the allocated memory 598.11 MiB is allocated by PyTorch, and 341.89 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
!zip -r pasta_compactada.zip /content/runs/detect/embrapac_runs/

from google.colab import files
files.download('pasta_compactada.zip')


# Teste do modelo

## Testando localmente em Ubuntu 24.04 com AMD Radeon

A partir do arquivo de saída do modelo já treinado (`/content/runs/detect/train/weight/best.pt`) parece possível rodar a inferência num PC com placa AMD na expectativa de conseguir testar detecções em imagens/vídeos a ~5-15 FPS dependendo da resolução.

A identificação da GPU do PC local foi obtido da seguinte forma:

```
$ lspci | grep -i vga
06:00.0 VGA compatible controller: Advanced Micro Devices, Inc. [AMD/ATI] Renoir [Radeon RX Vega 6 (Ryzen 4000/5000 Mobile Series)] (rev d3)
```

### Criando ambiente Python local

Virtual Environment criado para python 3.12.3: `pyenv/yolo`

Instalar as seguintes dependências para testar o modelo localmente:

`pip install ultralytics opencv-python`

Depois, para inferência (testar em imagens ou vídeo), você usa esse best.pt:

```
from ultralytics import YOLO

model = YOLO("best.pt")  # caminho do best.pt que você treinou

# testar em uma imagem
results = model("alguma_imagem.jpg", save=True)

# ou em vídeo
results = model("video.mp4", save=True)
```

In [ ]:
# CÓDIGO PARA TESTAR UM MODELO TREINADO LOCALMENTE, O QUE CONSIDERA O best.pt CARREGADO MANUALMENTE AQUI

from ultralytics import YOLO

# trained_model = "/content/best.pt"
#trained_model = "/content/runs/detect/train/weights/best.pt"
trained_model = f"/content/runs/detect/embrapac_runs/{dataset_name}/weights/best.pt"
#trained_model = "/content/best.pt"

model = YOLO(trained_model)

# classes do dataset
print('-------------------- CLASSES DO MODELO')
print(model.names)
print('--------------------')

# Sugestão pelo V3 a partir da análise dos gráficos Box:
# Para começar a inferência no Raspberry Pi 5: confidence entre 0,25 e 0,35
# Prioridade for evitar disparos falsos no background, subir para algo como 0,40–0,50

# results = model("/content/caixa_com_etiqueta3.jpg", save=True)  # save=True salva as imagens anotadas em runs/detect/predict/
results = model("/content/IMG_20260309_100115.jpg", conf=0.5) # coloca a confiabilidade em 10% para aceitar melhor imagens que não casam perfeitamente com os dados de treinamento

# print(results)
# if results is None:
#   print("Nenhum resultado encontrado")
#   exit()

# results é uma LISTA de Results (1 item se for 1 imagem)
print(f"Processou {len(results)} frames/imagens")

# Pega o primeiro resultado (results[0])
r = results[0]

# 1. Ver a imagem anotada (abre janela)
r.show()

# 2. Extrair as detecções manualmente (para lógica customizada)
if r.boxes is not None:  # se achou objetos
    print(f"Achou {len(r.boxes)} objetos")

    for box in r.boxes:
        # Coordenadas da bbox: [x1, y1, x2, y2] em pixels
        xyxy = box.xyxy[0].tolist()
        print(f"Bbox: {xyxy}")

        # Classe (0=classe1, 1=classe2, etc.) - confere no seu data.yaml
        cls_id = int(box.cls[0])
        print(f"Classe ID: {cls_id}")

        # Confiança (0-1)
        conf = float(box.conf[0])
        print(f"Confiança: {conf:.2f}")

        if conf > 0.5:  # threshold típico
            print("→ OBJETO VÁLIDO DETECTADO")


# Modo 2: Realizando fotos e rotulando

Nesse modo manual o trabalho será tirar a foto dos objetos que irão simular os devidos objetos da linha de produção.

Devem ser definidos 3 tamanhos de caixas e estabelecer a estratégia de tirar as fotos como se as caixas estivessem na linha de produção, de repente alinhando os diferentes tamanhos para na sequência serem rotuladas numa mesma foto para o modelo identificar os diferentes tamanhos possíveis.

Para o caso do rótulo, podemos ter um rótulo padrão


**Rotulagem**

A seguinte ferramenta pode ser utilizada para rotular as imagens das fotos capturadas.
https://github.com/HumanSignal/labelImg

